# GeoBench Jupyter Notebook Example

This notebook demonstrates how to use GeoBench to benchmark code execution in Jupyter notebooks.

First we define a function to benchmark:

In [1]:
import math

def count_primes(n):
    count = 0
    for i in range(2, n):
        if all(i % j != 0 for j in range(2, int(math.sqrt(i)) + 1)):
            count += 1
    return count

## Method 1: Using the Geobench Class

You can use the `Geobench` class to manually start and finish benchmarking around code execution.

In [2]:
from geobench import Geobench

# Create a benchmark instance
bench = Geobench(
    name="count-primes",
    run_monitor=2.0,     # Monitor for 2 seconds before and after execution
    clean=True,          # Clean output directory if it exists
)

In [3]:
# Start benchmarking
bench.start("count-primes-1M")

# Execute code to benchmark
try:
    result = count_primes(1_000_000)
    print(f"Found {result} prime numbers.")
    success = True

except Exception as err:
    print(f"Error: {err}")
    success = False

# Finish benchmarking
summary = bench.stop(success)

Starting benchmark: count-primes-1M
Storing system information.
Clearing system caches.
Waiting 2.0 s before the run.
Baseline monitoring for 2.0 s.
Process monitoring started.
Found 78498 prime numbers.
Process monitoring stopped.
Waiting 2.0 s after the run.
Endline monitoring for 2.0 s.
Benchmark completed in 7.72 s.


In [4]:
# Generate HTML report
report_path = bench.generate_report()
print(f"Report generated at: {report_path}")

Report generated at D:\Personal\projects\python\geobench\examples\count-primes\report.html
Report generated at: D:\Personal\projects\python\geobench\examples\count-primes\report.html


## Method 2: Using the @geobench decorator

For a simpler approach, you can use the `@geobench` decorator.

Because this example uses multi-processing, `count_primes()` method defined above cannot be used directly. Instead, we import an identical function from `count_primes.py` file.

In [5]:
from geobench import geobench

from count_primes import count_primes

# Define a function with the benchmark decorator
@geobench(name="parallel-count-primes", clean=True)
def parallel_count_primes(n, cores=4):
    import multiprocessing
    
    with multiprocessing.Pool(cores) as pool:
        results = pool.map(count_primes, [n]*cores)

    return sum(results)

In [6]:
result = parallel_count_primes(1_000_000, cores=4)
print(f"Found in total {result} prime numbers.")

Starting benchmark: parallel_count_primes
Storing system information.
Clearing system caches.
Waiting 2.0 s before the run.
Baseline monitoring for 2.0 s.
Process monitoring started.
Executing the function with process monitoring.
Process monitoring stopped.
Waiting 2.0 s after the run.
Endline monitoring for 2.0 s.
Benchmark completed in 7.60 s.
Report generated at D:\Personal\projects\python\geobench\examples\parallel-count-primes\report.html
Found in total 313992 prime numbers.
